# Rung 0 — the reliability of the assay, in plain language

**Task** `rung0-assay-reliability` · **Spec** [design.md](design.md) ·
**Audit** [audit.md](audit.md) · **Verification** [verification.md](verification.md)

Run this notebook top to bottom. Every number below is read from a committed table when you run
it, so what you see is what the artifacts say — not what anyone typed.

---

## The question

How much of a measured drug response is signal rather than assay noise?

A model that predicts these responses cannot agree with the measurement better than the
measurement agrees with itself. That self-agreement is the ceiling every later rung is read
against: a score of 0.2 against a ceiling of 0.2 sits at the limit the assay supports, while the
same 0.2 against a ceiling of 0.9 is a large shortfall, and reporting a rung without its ceiling
makes those two indistinguishable.

## How it is measured

Each cell line and drug was screened on several plates. Split those plates into two groups,
average each group's per-gene log2 fold change, and correlate the two averaged profiles across
genes. Do that for every condition and take the mean. Spearman-Brown then lifts that half-data
correlation to the reliability of the full measurement.

Two gene sets, because they answer different questions:

- **All genes.** Most genes do not respond to most drugs, so this is largely a measure of how
  reproducibly the assay reports a flat profile.
- **Responders.** Only the genes that condition's **first** plate group called differentially
  expressed. Chosen from the first group and scored against the second, so no gene is picked
  using the half it is judged on.

And one decomposition: the screen publishes a standard error for every fold change, but that
error only sees cell sampling within a plate. Splitting each delta's variance into its
between-plate and within-plate parts says whether the noise a model actually meets is plate
effects or cell sampling.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

TASK = Path("docs/tasks/rung0-assay-reliability")
if not TASK.exists():                      # allow running from inside the task folder
    TASK = Path(".")
FIG = TASK / "figures"

def table(name, **kw):
    """Read a committed table, or say plainly that the run has not happened yet."""
    p = TASK / name
    if not p.exists():
        print(f"[not present yet] {p}")
        return None
    return pd.read_csv(p, **kw)

def show(name, caption=""):
    p = FIG / name
    if not p.exists():
        print(f"[figure not present yet] {p}")
        return
    if caption:
        print(caption)
    display(Image(filename=str(p)))

summary = table("rung0_reliability.csv")
S = summary.iloc[0].to_dict() if summary is not None else {}
def g(k, default=float("nan")):
    return S.get(k, default)
print("run present:" , summary is not None)

## The hypotheses, stated before the run

From `design.md`, "Expected result". Each is answered below with the number that settles it.

1. Correlations over **all genes** will be low, since most genes are not affected by the drugs.
2. Correlations over the **responder** genes will be **higher** than over all genes.
3. Noise will be higher in the cell line and drug combinations with lower correlations.
4. Aggregate noise will be **higher** in the responder genes than over all genes.

In [ ]:
if summary is not None:
    rows = []
    for fam, label in (("all", "all genes"), ("responder", "responders")):
        rows.append({
            "gene set": label,
            "conditions scored": int(g(f"{fam}_n_pairs")),
            "split-half r (mean)": g(f"{fam}_splithalf_mean_r"),
            "median": g(f"{fam}_splithalf_median_r"),
            "quartiles": (
                f'{g(f"{fam}_splithalf_q1_r")} - {g(f"{fam}_splithalf_q3_r")}'
            ),
            "Spearman-Brown ceiling": g(f"{fam}_spearman_brown_full"),
            "SB, equal-half conditions": g(f"{fam}_spearman_brown_full_even_plates"),
            "conditions equal-half": int(g(f"{fam}_n_pairs_even")),
            "vs different-drug floor": g(f"{fam}_null_diff_drug_mean_r"),
            "vs same-drug floor": g(f"{fam}_null_same_drug_mean_r"),
            "p vs different-drug": g(f"{fam}_p_vs_null"),
            "p vs same-drug": g(f"{fam}_p_vs_same_drug"),
            "MDE (80% power)": g(f"{fam}_mde_80_vs_diff_drug"),
        })
    display(pd.DataFrame(rows).set_index("gene set").T)

In [ ]:
if summary is not None:
    a, r = g("all_splithalf_mean_r"), g("responder_splithalf_mean_r")
    print(f"Hypothesis 2 -- responders above all genes: "
          f"{'HELD' if r > a else 'DID NOT HOLD'} ({r:.3f} vs {a:.3f}, difference {r - a:+.3f})")
    for fam, label in (("all", "all genes"), ("responder", "responders")):
        clears = g(f"{fam}_splithalf_mean_r") > max(
            g(f"{fam}_null_diff_drug_mean_r"), g(f"{fam}_null_same_drug_mean_r"))
        p_diff, p_same = g(f"{fam}_p_vs_null"), g(f"{fam}_p_vs_same_drug")
        print(f"{label}: clears both floors: {clears}; "
              f"p = {p_diff} (different-drug), {p_same} (same-drug)")

---

## Step 1 — build: what the screen actually contains

Before any statistic, what is in the pool. The composition panels are the denominator every
later count is a subset of; the fold-change panel puts the real screen beside a synthetic pool
with a planted answer, so the control is visibly the same shape as the data it stands in for
rather than assumed to be.

In [ ]:
pool = table("rung0_pool_description.csv")
if pool is not None:
    print(f"conditions in the pool: {len(pool):,}")
    print(f"cell lines: {pool['patient'].nunique()}   drugs: {pool['drug'].nunique()}")
    print(f"conditions with at least two plates (splittable): {(pool['n_plates'] >= 2).sum():,}")
    print(f"conditions whose plates split into equal halves: {int(pool['n_plates_even'].sum()):,}")
    display(pool["n_plates"].value_counts().sort_index().rename("conditions").to_frame().T)
show("01_build.png")

## Step 2 — split: one condition becomes two half-profiles

Plates are assigned to groups by a hash of the plate identifier, so the split is fixed and
carries no random seed. Most conditions have three plates, which means one plate against two —
unequal halves, which is why the Spearman-Brown correction is reported again over the
even-plate conditions where it is exact.

In [ ]:
per_pair = table("rung0_per_pair_r.csv")
if per_pair is not None:
    print(f"conditions with a scoreable all-gene correlation: {per_pair['r'].notna().sum():,}")
    print(f"median genes scored per condition: {per_pair['n_genes_scored'].median():,.0f}")
show("02_split.png")

## Step 3 — select: which genes count as responders

A responder is a gene the condition's **first** plate group called differentially expressed.
Selection reads that group alone and the correlation is still first group against second.

Panel (c) is why. Both bars come from a pool with no signal at all. Selecting on the pooled data
— the natural mistake of calling differential expression on every plate at once and then
correlating the halves — keeps the genes whose noise agreed, and reads as reproducibility that
nothing generated. The gap between the bars is the bias the one-sided rule avoids.

Panel (d) is how far the two groups agree on which genes responded. It is a diagnostic and never
an input: keeping the genes both groups called *is* the pooled rule panel (c) measures.

In [ ]:
leak = table("rung0_leakage_control.csv")
if leak is not None:
    display(leak)
    one = float(leak.loc[leak["rule"] == "one-sided", "mean_r"].iloc[0])
    pooled = float(leak.loc[leak["rule"] == "pooled", "mean_r"].iloc[0])
    print(f"\nOn signal-free data the shipped one-sided rule reads {one:+.3f}; "
          f"selecting on the pooled data reads {pooled:+.3f}.")
    print(f"The inflation avoided is {pooled - one:+.3f}.")

ov = table("rung0_responder_overlap.csv")
if ov is not None:
    print(f"\nresponders per condition, first group: median {ov['n_first'].median():,.0f}")
    print(f"Jaccard overlap of the two groups' responder sets: median {ov['jaccard'].median():.3f}")
if per_pair is not None and "n_responders" in per_pair:
    med = per_pair["n_responders"].median()
    print(f"responders actually scored per condition: median {med:,.0f}")
show("03_select.png")

### A property of the selection rule, stated plainly

The rule is "significant in **at least one** of the first group's rows". Under the null a gene is
therefore admitted with probability `1 - (1 - alpha)^k` over `k` rows, not `alpha` — with several
plates and three doses in the first group that is well above five percent.

This does not bias the responder reliability: the mismatched-pair nulls apply the same rule to
the same group, so the comparison stays like for like. Its effect is directional and works
*against* the hypothesis — a larger responder set under the null dilutes the responder statistic
toward the all-gene one. Whatever separation is reported above is therefore conservative.

## Step 4 — score: the two reliabilities

The scatters are individual conditions, drawn twice: over all genes, then over that condition's
responders. Each panel's correlation is recomputed from the points plotted and written to a
companion file, so the printed number is checkable rather than asserted.

The histograms beneath are every condition's correlation, with the control pools underneath on
shared axes — a pool with a planted reliability, and a pool with none.

In [ ]:
idx = table("rung0_example_pair_index.csv")
if idx is not None:
    display(idx)
show("04_score.png")

## Step 5 — decompose: what kind of noise the ceiling is made of

The screen's own `lfcSE` is the standard error of one plate's treated-versus-control contrast: it
sees cell sampling and cannot see plate-to-plate variation. For each gene, dose and condition
with at least two plates, the variance of the fold change across plates has expectation
`sigma^2_plate + mean(lfcSE^2)`, so subtracting the mean squared standard error leaves the plate
component alone. Dose is held fixed, so a dose effect cannot be charged to plate noise.

If the plate component dominates, the published standard errors understate assay noise and the
split-half is the only honest ceiling. If it is near zero, the assay's noise is cell sampling —
and a later rung could in principle read against a ceiling derived from `lfcSE` on material with
no replicate plates at all, which is the only route to a ceiling for unreplicated samples.

In [ ]:
noise = table("rung0_noise_decomposition.csv")
if noise is not None:
    n = noise.iloc[0]
    display(noise.T.rename(columns={0: "value"}))
    print(f"\nMean between-plate share of delta variance: {n['between_plate_fraction_mean']:.3f}")
    print(f"Gene-conditions where plate effects dominate (share > 0.5): "
          f"{n['frac_plate_dominated']:.1%}")
    verdict = ("plate effects dominate, so the published standard errors understate assay noise "
               "and the split-half is the only honest ceiling"
               if n["between_plate_fraction_mean"] > 0.5 else
               "cell sampling dominates, so the published standard errors and the split-half "
               "describe substantially the same noise")
    print(f"\nReading: {verdict}.")

# The stratified view, computed in the engine over EVERY gene-condition rather than over the
# sample the figure draws. If the between-plate share moves with expression or with response
# size, the single headline number above is an average over conditions that differ.
strata = table("rung0_noise_strata.csv")
if strata is not None:
    def weighted(df, key):
        out = {}
        for level, part in df.groupby(key):
            w = part["n"].to_numpy(dtype=float)
            v = part["between_plate_fraction_mean"].to_numpy(dtype=float)
            out[int(level)] = round(float((v * w).sum() / w.sum()), 4)
        return pd.Series(out, name="between-plate share")

    print("\nby expression quartile (1 = lowest baseMean):")
    display(weighted(strata, "expression_quartile").to_frame().T)
    print("by response-size quartile (1 = smallest mean absolute log2 fold change):")
    display(weighted(strata, "response_quartile").to_frame().T)
show("05_decompose.png")

## Step 6 — null: what the reliabilities are read against

A split-half correlation has a floor above zero, because genes share structure whether or not two
profiles come from the same perturbation. Three floors are built by pairing one condition's first
group with a *different* condition's second group:

- **any pair** — reported for continuity only.
- **different drug and line** — the generic-structure floor a ceiling must clear to be a ceiling.
- **same drug, different line** — the stricter, line-specificity floor: two lines given one drug
  already share that drug's generic response, so clearing this says the response is specific to
  the line rather than to the compound.

A mismatched draw for the responder statistic uses the *first* condition's responder genes — the
same selection rule as a matched pair, so the null answers the same question.

In [ ]:
nulls = table("rung0_null_draws.csv")
if nulls is not None:
    display(nulls.groupby(["gene_set", "stratum"])["r"].agg(["count", "mean", "median"]).round(4))
show("06_null.png")

In [ ]:
for gene_set, fname in (("all genes", "rung0_permutation_summary.csv"),
                        ("responders", "rung0_permutation_summary_responder.csv")):
    perm = table(fname)
    if perm is not None:
        print(f"--- permutation check, {gene_set} ---")
        display(perm.T.rename(columns={0: "value"}))

The bootstrap that produces the p-values above treats the mismatched draws as independent, and
they are not: every draw reuses the same half-profiles. The permutation check carries that
dependence by construction — it permutes the pairing so no condition meets its own partner — and
reports the **design effect**, the ratio of the true sampling variance of the mean to the variance
an independent pool would have. A design effect near 1 means the bootstrap was not misled; a large
one means its p-values are optimistic by that factor in variance.

## The controls, and what they establish

Every measurement step ships a positive control that plants a known answer and requires the
shipped code to recover it, and a negative control that feeds signal-free data and requires null.
These run in continuous integration, not only here.

In [ ]:
terc = table("rung0_effect_terciles.csv")
if terc is not None:
    display(terc)
    rising = terc["mean_r"].is_monotonic_increasing
    print(f"\nEmpirical control -- reproducibility rises with response size: "
          f"{'HELD' if rising else 'DID NOT HOLD'}")
    print("An assay that cannot find more reproducibility where there is more signal is broken.")
show("07_terciles.png")

In [ ]:
mde = table("rung0_mde_curve.csv")
if mde is not None and summary is not None:
    obs = mde[mde["observed"]]
    display(obs)
    print("\nThe smallest effect this screen could have detected at 80% power, at its own")
    print("condition count. Reported so a null result cannot be confused with an underpowered")
    print("one -- the distinction the organoid rung will turn on, with a tenth of the conditions.")
show("08_power.png")

## Conclusions

Read the four hypotheses against the numbers above. Where one did not hold, that is the finding,
not a defect to explain away.

In [ ]:
if summary is not None:
    a, r = g("all_splithalf_mean_r"), g("responder_splithalf_mean_r")
    sb_a, sb_r = g("all_spearman_brown_full"), g("responder_spearman_brown_full")
    print("1. All-gene correlations are low:", f"mean r = {a:.3f} (ceiling {sb_a:.3f})")
    print("2. Responders exceed all genes:",
          f"{'YES' if r > a else 'NO'} -- {r:.3f} vs {a:.3f} (ceiling {sb_r:.3f})")
    if noise is not None:
        print("4. Aggregate noise higher in responders: see the decomposition above")
    print()
    print("What a later rung divides by:")
    print(f"  scoring all genes      -> {sb_a:.3f}")
    print(f"  scoring responders     -> {sb_r:.3f}")
    print("Use whichever matches the genes that rung scores. Rung 0 measures at full extent;")
    print("a rung scoring a subset declares and computes its own restriction of these ceilings.")

## What this rung does not establish

- A **dose-resolved** reliability. Doses are pooled, so a condition means "this drug at this
  screen's dose design".
- Sensitivity of either number to the **choice of split**. One hash-defined split is used.
- Any **restriction** of these ceilings to a later rung's genes or drugs. Each later rung
  declares and computes its own, before it scores anything.

## Scripts this task touched

`scripts/delta_reproducibility.py` (build, split, select, score, decompose, null, exports,
figures) · `scripts/permutation_null.py` (the permutation check) ·
`src/fmharness/statistics.py` (significance, power, Spearman-Brown) ·
`src/fmharness/figures.py` · `src/fmharness/synthetic.py` (the planted control pools) ·
`scripts/verify_rung0.py` and [verify.ipynb](verify.ipynb) (the claim-by-claim recomputation) ·
`scripts/alpine/delta_reproducibility.sbatch`, `scripts/alpine/permutation_null.sbatch`.

Controls and known-answer tests: `tests/test_rung0_controls.py`, `tests/test_rung0_figures.py`,
`tests/test_statistics_known_answers.py`, `tests/test_permutation_null.py`,
`tests/test_verify_rung0.py`.